# Receptor refinement geometry — before vs after the signed graph

Our thesis is that the signed bipartite GNN **improves the receptor representation from
binding data**. This notebook makes that visible. It takes the M2OR cold-molecule
(*inductive*) signed-graph run — five trained `_SignedSage` models, one per seed
(folds 42–46) — and shows, in rotatable 3D (plotly):

1. the **initial** receptor cloud — raw mean-pooled ESM-1b, before any message passing;
2. **five refined** clouds — the graph-enriched receptor vectors `z[PROT]`, one per model.

Every point is a receptor, coloured by its **OR family** (~13 colours). If the graph
reorganises receptors by their ligand profile, the refined clouds should sharpen family
structure relative to raw ESM. The analysis cells at the end quantify that
(silhouette / kNN purity), check whether the five models agree on the geometry (CKA),
and rank which receptors move most toward their family.


In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)   # orbind resolves its data/ paths relative to CWD, so run from the repo root

import re
import numpy as np
import pandas as pd
import torch

try:
    import plotly.graph_objects as go
    import plotly.express as px
except ModuleNotFoundError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"], check=True)
    import plotly.graph_objects as go
    import plotly.express as px

from orbind.regimes import full_full_pairs, load_split
from orbind.gnn_extractor import (
    GnnSignedExtractor, _build_universe, _mp_edges, _edge_index_dict,
    _train_one, MOL, PROT,
)

DATA   = ROOT / "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT, "| device:", DEVICE)

In [ ]:
# --- which run's models to visualise -------------------------------------
# M2OR cold-molecule (inductive) signed-graph run: 5 seeds (folds 42-46), each a
# trained _SignedSage. If the run saved checkpoints under
# RUN_DIR/checkpoints/repeat_{seed}/gnn_{CLS_NAME}_model0.pt they are LOADED; else the
# notebook REPRODUCES each model from the exact same config/protocol (the graph is tiny,
# ~seconds/seed), so it runs anywhere the embeddings live.
RUN_DIR  = ROOT / "results/ensemble_logs/m2or-inductive-chemberta-fixed/gnn_signed_q99_greedy_clsprotmol"
CLS_NAME = "cls"                      # source name -> checkpoint file gnn_{CLS_NAME}_model0.pt
REGIME   = "inductive_molecule_v5"
SEEDS    = [42, 43, 44, 45, 46]

# graph config = the run's `cls=gnn_signed` source, verbatim (q99 / greedy / emit=prot)
PROT_NPZ = DATA / "embeddings/proteins/esm1b_650m_mean.npz"
MOL_NPZ  = DATA / "embeddings/molecules/gin_supervised_contextpred_all_m2or.npz"
GRAPH_KW = dict(q=0.99, criterion="greedy_pair_cover", edge_threshold=0.0,
                k_mode="coverage_quantile", emit="prot", n_models=1, hidden=256)

N_FAMILY_COLORS = 13   # keep the top (N-1) OR families by count + one "other" bucket

## 1 · Load / reproduce the five models and pull the embeddings

For each seed we rebuild the exact inductive universe and message-passing graph, load the
trained encoder (or retrain it), then read off two things: the **initial** node features
(raw ESM, shared across seeds) and the **refined** receptor vectors `z[PROT]` (one set per
seed). Receptors are keyed by name so everything aligns across seeds.


In [ ]:
pairs = full_full_pairs()
ext = GnnSignedExtractor(name=CLS_NAME, protein_path=str(PROT_NPZ),
                         molecule_path=str(MOL_NPZ), **GRAPH_KW)

initial, refined = None, {}
for seed in SEEDS:
    tr, va, te = load_split(REGIME, seed)
    # keep only rows this graph covers (protein + molecule embedding present) --
    # the run does this via `--on-missing drop` before building the universe.
    tr, va, te = (i[ext.covered(pairs, i)] for i in (tr, va, te))
    all_idx = np.concatenate([tr, va, te])
    mol_to_i, prot_to_i, x_mol, x_prot = _build_universe(pairs, all_idx, ext._proteins, ext._molecules)
    pos, neg = _mp_edges(pairs, tr, mol_to_i, prot_to_i, ext.q, ext.criterion,
                         ext.edge_threshold, ext.k_mode, ext.task)
    pos_eidx, neg_eidx = _edge_index_dict(pos, neg)

    tdf = pairs.iloc[tr]
    mi = tdf["inchikey"].map(mol_to_i).to_numpy()
    pi = tdf["receptor"].map(prot_to_i).to_numpy()
    y  = tdf["label"].to_numpy(dtype=np.float32)

    ckpt = RUN_DIR / "checkpoints" / f"repeat_{seed}" / f"gnn_{CLS_NAME}_model0.pt"
    z_mol, z_prot, model = _train_one(
        ext._build_model, x_mol, x_prot, pos_eidx, neg_eidx, mi, pi, y,
        ext._hp(seed), DEVICE, checkpoint_path=(ckpt if ckpt.exists() else None))

    refined[seed] = {r: z_prot[i] for r, i in prot_to_i.items()}
    if initial is None:
        initial = {r: x_prot[i].numpy() for r, i in prot_to_i.items()}
    print(f"seed {seed}: {'loaded' if ckpt.exists() else 'reproduced':10s} "
          f"| {len(prot_to_i):4d} receptors | {len(pos[0])}+/{len(neg[0])}- MP edges "
          f"| refined dim {z_prot.shape[1]}")

REC = sorted(set(initial).intersection(*[set(refined[s]) for s in SEEDS]))
init_mat     = np.stack([initial[r] for r in REC])
refined_mats = {s: np.stack([refined[s][r] for r in REC]) for s in SEEDS}
print(f"\naligned on {len(REC)} receptors | init {init_mat.shape} | refined {refined_mats[SEEDS[0]].shape}")

## 2 · Colour by OR family

Same recipe as `protein_embeddings/pca19_family_3d.ipynb`: receptor → BW reference entry →
leading `OR<n>` token. We keep the top `N_FAMILY_COLORS-1` families by size and fold the
long tail into `other`, so the legend stays ~13 colours.


In [ ]:
bw = pd.read_csv(DATA / "processed/bw_ref_used_curated.csv")
seq2ref = bw.set_index("receptor")["ref_entry"].to_dict()

def or_family(rec):
    slug = seq2ref.get(rec, "unk_human").replace("_human", "")
    m = re.match(r"^(or?\d+)", slug)
    return m.group(1).upper() if m else slug.upper()

fam_all = {r: or_family(r) for r in REC}
counts  = pd.Series(list(fam_all.values())).value_counts()
top     = list(counts.index[:N_FAMILY_COLORS - 1])
fam     = np.array([fam_all[r] if fam_all[r] in top else "other" for r in REC])

fam_levels = [f for f in top if f in set(fam)] + (["other"] if "other" in fam else [])
palette   = px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
color_map = {f: palette[i % len(palette)] for i, f in enumerate(fam_levels)}
if "other" in color_map:
    color_map["other"] = "#c0c0c0"
print(f"{len(fam_levels)} colour groups:", fam_levels)
counts.head(N_FAMILY_COLORS)

## 3 · Initial cloud — raw ESM-1b

The starting point every model shares: mean-pooled ESM-1b, PCA-projected to 3D. Drag to
rotate; hover shows the receptor name.


In [ ]:
from sklearn.decomposition import PCA

def to3d(M):
    return PCA(n_components=3, random_state=0).fit_transform(np.asarray(M, dtype=np.float64))

def cloud_fig(coords, title):
    fig = go.Figure()
    for f in fam_levels:
        m = fam == f
        if not m.any():
            continue
        names = [REC[i] for i in np.where(m)[0]]
        fig.add_trace(go.Scatter3d(
            x=coords[m, 0], y=coords[m, 1], z=coords[m, 2], mode="markers", name=f,
            marker=dict(size=3.5, color=color_map[f], opacity=0.85),
            text=names, hovertemplate="%{text}<extra>" + f + "</extra>"))
    fig.update_layout(title=title, height=650, legend_title="OR family",
                      margin=dict(l=0, r=0, t=45, b=0),
                      scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"))
    return fig

cloud_fig(to3d(init_mat), f"Initial receptors — raw ESM-1b ({init_mat.shape[1]}d) &#8594; PCA-3").show()

## 4 · Refined clouds — one per model (5 seeds)

The graph-enriched receptor vectors. Five independently-seeded encoders, so five clouds —
compare their family organisation to the raw cloud above (and to each other).


In [ ]:
for s in SEEDS:
    cloud_fig(to3d(refined_mats[s]),
              f"Refined receptors — seed {s} ({refined_mats[s].shape[1]}d) &#8594; PCA-3").show()

## 5 · Does refinement sharpen family structure?

Two label-aware geometry scores on the **full-dimensional** vectors (not the 3D
projection), computed over real families only (the `other` bucket dropped): the
**silhouette** of the family clustering, and **kNN purity** (share of each receptor's 5
nearest neighbours in the same family). Higher = families sit together more cleanly. We
compare raw ESM against each refined model.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

keep = fam != "other"

def knn_purity(X, labels, k=5):
    k = min(k, len(X) - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    idx = nn.kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([np.mean(labels[nbrs] == labels[i]) for i, nbrs in enumerate(idx)]))

def geo_scores(X):
    Xk, lk = X[keep], fam[keep]
    return silhouette_score(Xk, lk), knn_purity(Xk, lk)

rows = [("initial (raw ESM)", *geo_scores(init_mat))]
rows += [(f"refined seed {s}", *geo_scores(refined_mats[s])) for s in SEEDS]
geo = pd.DataFrame(rows, columns=["embedding", "silhouette", "knn_purity_k5"])

ref = geo.iloc[1:]
print(f"raw ESM: silhouette {geo.iloc[0].silhouette:+.3f}, kNN purity {geo.iloc[0].knn_purity_k5:.3f}")
print(f"refined: silhouette {ref.silhouette.mean():+.3f} +/- {ref.silhouette.std():.3f}, "
      f"kNN purity {ref.knn_purity_k5.mean():.3f} +/- {ref.knn_purity_k5.std():.3f} (mean over 5 seeds)")
display(geo.style.format({"silhouette": "{:+.3f}", "knn_purity_k5": "{:.3f}"}).hide(axis="index"))

fig = go.Figure()
fig.add_bar(x=geo.embedding, y=geo.knn_purity_k5, marker_color=["#888"] + ["#4c78a8"] * len(SEEDS),
            name="kNN purity")
fig.add_hline(y=geo.iloc[0].knn_purity_k5, line_dash="dot", line_color="#888",
              annotation_text="raw ESM")
fig.update_layout(title="kNN family purity (k=5): raw ESM vs 5 refined models",
                  yaxis_title="purity", height=420, margin=dict(t=45)).show()

## 6 · Do the five models agree on the geometry?

Linear **CKA** between every pair of embedding spaces (representation-similarity that is
invariant to rotation/scaling and works across different dimensionalities). If the five
seeds discover the *same* refined geometry — not seed noise — the refined-vs-refined block
should be high; the init-vs-refined column shows how far each model moved away from raw
ESM.


In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0)
    Y = Y - Y.mean(0)
    return float((np.linalg.norm(Y.T @ X) ** 2) /
                 (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)))

mats = {"init": init_mat, **{f"s{s}": refined_mats[s] for s in SEEDS}}
keys = list(mats)
C = np.array([[linear_cka(mats[a], mats[b]) for b in keys] for a in keys])

go.Figure(go.Heatmap(z=C, x=keys, y=keys, colorscale="Viridis", zmin=0, zmax=1,
                     text=np.round(C, 2), texttemplate="%{text}", showscale=True)
          ).update_layout(title="Linear CKA — geometry agreement (raw init vs 5 refined models)",
                          height=520, margin=dict(t=45)).show()

rr = C[1:, 1:][np.triu_indices(len(SEEDS), 1)]
print(f"refined<->refined CKA: {rr.mean():.3f} +/- {rr.std():.3f}  |  "
      f"init<->refined CKA: {C[0, 1:].mean():.3f} +/- {C[0, 1:].std():.3f}")

## 7 · Which receptors move most toward their family?

Per receptor, the cosine similarity to its own family centroid, raw vs refined (averaged
over the five models). A positive `delta` means the graph pulled that receptor closer to
its family — the receptors where ligand-profile refinement adds the most that raw sequence
identity missed.


In [ ]:
def centroid_cos(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    sim = np.full(len(REC), np.nan)
    for f in fam_levels:
        if f == "other":
            continue
        m = fam == f
        if m.sum() < 2:
            continue
        c = Xn[m].mean(0)
        c = c / (np.linalg.norm(c) + 1e-9)
        sim[m] = Xn[m] @ c
    return sim

base     = centroid_cos(init_mat)
ref_mean = np.nanmean(np.stack([centroid_cos(refined_mats[s]) for s in SEEDS]), axis=0)
mv = (pd.DataFrame({"receptor": REC, "family": fam, "cos_init": base,
                    "cos_refined": ref_mean, "delta": ref_mean - base})
      .query("family != 'other'").dropna().sort_values("delta", ascending=False))

print(f"mean cosine-to-family-centroid: raw {np.nanmean(base):.3f} -> refined "
      f"{np.nanmean(ref_mean):.3f}  (mean delta {mv['delta'].mean():+.3f})")
display(pd.concat([mv.head(10), mv.tail(5)])
        .style.format({"cos_init": "{:.3f}", "cos_refined": "{:.3f}", "delta": "{:+.3f}"})
        .hide(axis="index"))

**Read the three analyses together.** If silhouette/kNN purity rise from raw to refined,
the graph is genuinely tightening family structure; the CKA block says whether the five
seeds agree that this is a real, reproducible geometry rather than per-seed noise; and the
mover table names the receptors that benefit most — a shortlist of where binding-profile
information corrects sequence-only identity. This is the receptor-side improvement the
paper argues the field has under-explored, made visible.
